In [20]:
import random
import numpy as np

random.seed(189)
np.random.seed(189)

import matplotlib.pyplot as plt
from scipy import io
import pandas as pd

from scipy.stats import multivariate_normal
from sklearn.preprocessing import StandardScaler

In [21]:
# Load data
spam = np.load("/kaggle/input/d/phoenixpham10/spam-data-hw3/spam-data-hw3.npz") # May have to change file path
spam_training_data, spam_training_labels = spam['training_data'], spam['training_labels']
spam_test_data = spam['test_data']

In [22]:
# Function that shuffles data given a dataset and its labels
def shuffle_data_labels(data, labels):
    data_points = np.arange(data.shape[0])
    np.random.shuffle(data_points)
    return [data[data_points], labels[data_points]]
    
# Shuffle dataset
spam_training_data, spam_training_labels = shuffle_data_labels(spam_training_data, spam_training_labels)

# Partition the shuffled dataset
spam_training_data_len = spam_training_data.shape[0]
spam_training_set_data, spam_training_set_labels = spam_training_data[:int(0.8 * spam_training_data_len), :], spam_training_labels[:int(0.8 * spam_training_data_len)]
spam_validation_set_data, spam_validation_set_labels = spam_training_data[int(0.8 * spam_training_data_len):, :], spam_training_labels[int(0.8 * spam_training_data_len):]

In [23]:
scaler = StandardScaler()
spam_training_set_data_scaled = scaler.fit_transform(spam_training_set_data)
spam_validation_set_data_scaled = scaler.transform(spam_validation_set_data)

In [24]:
def normalize(X):
    centered_X = X - np.mean(X, axis=1, keepdims=True)

    norms = np.linalg.norm(centered_X, axis=1, keepdims=True) + 1e-8

    return centered_X / norms

In [25]:
spam_training_set_data_scaled = normalize(spam_training_set_data_scaled)
spam_validation_set_data_scaled = normalize(spam_validation_set_data_scaled)

In [26]:
def pooled_within_class_covariance(X, y, means):
    n, d = X.shape
    pooled_cov = np.zeros((d, d))

    classes = np.unique(y)

    for c_idx, c in enumerate(classes):
        class_data = X[y.ravel() == c]
        class_mean = means[c_idx]
        centered_data = class_data - class_mean
        pooled_cov += np.dot(centered_data.T, centered_data)

    return pooled_cov / n

def lda_log_posterior(X, means, pooled_cov, priors):
    n, d = X.shape
    c = len(means)
    log_post = np.zeros((n, c))

    pooled_cov = pooled_cov + 1e-8 * np.identity(d)

    for i in range(c):
        log_pdf = multivariate_normal.logpdf(X, mean=means[i], cov=pooled_cov, allow_singular=True)
        
        log_post[:, i] = log_pdf + np.log(priors[i])

    return log_post

In [27]:
def train_lda(X_train, y_train):
    classes = np.unique(y_train)

    means = []

    for c in classes:
        class_data = X_train[y_train.ravel() == c]
        means.append(np.mean(class_data, axis=0))

    pooled_cov = pooled_within_class_covariance(X_train, y_train, means)
    priors = np.bincount(y_train.ravel()) / len(y_train)

    return means, pooled_cov, priors

def predict_lda(X, means, pooled_cov, priors):
    log_post = lda_log_posterior(X, means, pooled_cov, priors)
    return np.argmax(log_post, axis=1)

def evaluate_lda(y_true, y_pred):
    error_rate = 1 - np.mean(y_pred == y_true.ravel())
    print(f"Error Rate: {error_rate:.4f}")
    return error_rate

def results_to_csv(y_test, file_name):
    y_test = y_test.astype(int)
    df = pd.DataFrame({'Category': y_test})
    df.index += 1
    df.to_csv(file_name, index_label='Id')

In [28]:
means, pooled_cov, priors = train_lda(spam_training_set_data_scaled, spam_training_set_labels)

y_val_pred = predict_lda(spam_validation_set_data_scaled, means, pooled_cov, priors)
print("Validation Set Performance:")
validation_error = evaluate_lda(spam_validation_set_labels, y_val_pred)

Validation Set Performance:
Error Rate: 0.0683


In [10]:
# Scale and normalize on full training set, and test set
spam_training_data_scaled = scaler.fit_transform(spam_training_data)
spam_training_data_scaled = normalize(spam_training_data_scaled)

spam_test_data_scaled = scaler.transform(spam_test_data)
spam_test_data_scaled = normalize(spam_test_data_scaled)

means, pooled_cov, priors = train_lda(spam_training_data_scaled, spam_training_labels)
y_test = predict_lda(spam_test_data_scaled, means, pooled_cov, priors)

results_to_csv(y_test, 'spam-submission.csv')
print("Submission file 'spam-submission.csv' has been created.")

Submission file 'spam-submission.csv' has been created.
